# Week 2 Lab: In-Context Learning & Sampling (Local Open-Weights)
**Research Engineer:** Victor Okoye
**Goal:** Systematically test how Few-Shot example count, formatting, and order affect an LLM's ability to extract JSON data. 
*Note: Executing locally using Hugging Face Transformers and Qwen2.5 (No API keys required).*

In [1]:
# Install the Hugging Face libraries needed to run models locally
!pip install transformers torch accelerate

## 1. The Test Environment (Local Model Setup)
Instead of pinging a remote API, we are downloading a 0.5-billion parameter model directly into Kaggle's GPU memory. We set the temperature very low (`0.1`) to force the model to be deterministic and mathematical, removing creative randomness.

In [2]:
import torch
from transformers import pipeline

# Download and load the model directly into Kaggle's GPU memory
# This takes about 30 seconds the very first time you run it.
print("Loading model weights...")
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)

def run_prompt_test(test_name, system_prompt, user_prompt, temp=0.1):
    print(f"\n--- Running Test: {test_name} (Temp: {temp}) ---")
    
    # Format the prompt using the model's standard chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Generate the response locally
    outputs = pipe(
        prompt, 
        max_new_tokens=100, 
        do_sample=True, 
        temperature=temp, 
        top_p=0.9
    )
    
    # Extract and print just the generated text
    generated_text = outputs[0]["generated_text"][len(prompt):]
    print(generated_text)

# The messy data we want the model to process
fixed_user_input = """
Hey, it's Amanda. I need to book a flight to Lagos on October 15th. 
My passport number is A1234567. 
"""

Loading model weights...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 2. Test 1: Zero-Shot (The Baseline)
We give the model the instructions, but zero examples of what success looks like.

In [3]:
zero_shot_system = """
Extract the user's Name, Destination, Date, and Passport. 
Output STRICTLY as JSON.
"""

run_prompt_test("Zero-Shot Baseline", zero_shot_system, fixed_user_input)

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Test: Zero-Shot Baseline (Temp: 0.1) ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


```json
{
  "Name": "Amanda",
  "Destination": "Lagos",
  "Date": "October 15th",
  "Passport": "A1234567"
}
```


## 3. Test 2: One-Shot (Adding the Pattern)
We provide exactly one example. Notice how the Transformer's self-attention mechanism instantly latches onto the exact JSON keys and structure we provide, applying them to the new data.

In [4]:
one_shot_system = """
Extract the user's Name, Destination, Date, and Passport. Output STRICTLY as JSON.

EXAMPLE:
User: "Yo I am Victor flying to Abuja on Nov 2. Pass: B98765"
Output: {"name": "Victor", "destination": "Abuja", "date": "Nov 2", "passport": "B98765"}
"""

run_prompt_test("One-Shot Pattern", one_shot_system, fixed_user_input)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Test: One-Shot Pattern (Temp: 0.1) ---
{"name": "Amanda", "destination": "Lagos", "date": "Oct 15", "passport": "A1234567"}


## 4. Test 3: The Recency Bias Test (Order Matters)
We give the model three examples, but sabotage the very last one by formatting it as a bulleted list instead of JSON. We are testing if the model's attention is hijacked by the most recently seen pattern.

In [5]:
recency_bias_system = """
Extract the user's Name, Destination, Date, and Passport. Output STRICTLY as JSON.

EXAMPLE 1:
User: "Victor to Abuja on Nov 2. Pass: B98765"
Output: {"name": "Victor", "destination": "Abuja", "date": "Nov 2", "passport": "B98765"}

EXAMPLE 2:
User: "Chinedu heading to London on Dec 1. Pass: C55555"
Output: {"name": "Chinedu", "destination": "London", "date": "Dec 1", "passport": "C55555"}

EXAMPLE 3 (Bad Format):
User: "Sarah to Kano on Jan 5. Pass: D11111"
Output: 
- Name: Sarah
- Dest: Kano
- Date: Jan 5
- Pass: D11111
"""

run_prompt_test("Recency Bias", recency_bias_system, fixed_user_input)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Test: Recency Bias (Temp: 0.1) ---
{
  "name": "Amanda",
  "destination": "Lagos",
  "date": "October 15",
  "passport": "A1234567"
}
